# K-MIMIC-MEDS — Validation du dataset

Ce notebook vérifie que le dataset produit par le pipeline ETL est conforme au format MEDS.

**Pipeline :**
1. `pre_meds.py` — transforme les `.xlsx` bruts en `.parquet` intermédiaires
2. `meds_convert.py` — convertit les `.parquet` intermédiaires en dataset MEDS final

**Structure attendue :**
```
data/output/
├── data/
│   ├── train/0.parquet
│   ├── tuning/0.parquet
│   └── held_out/0.parquet
└── metadata/
    ├── codes.parquet
    ├── dataset.json
    └── subject_splits.parquet
```

In [1]:
import json
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("data/output")

## 1. Structure des fichiers

In [2]:
# Vérifier que tous les fichiers attendus existent
expected = [
    "data/train/0.parquet",
    "data/tuning/0.parquet",
    "data/held_out/0.parquet",
    "metadata/codes.parquet",
    "metadata/dataset.json",
    "metadata/subject_splits.parquet",
]

for f in expected:
    path = OUTPUT_DIR / f
    status = "✅" if path.exists() else "❌ MANQUANT"
    print(f"{status}  {f}")

✅  data/train/0.parquet
✅  data/tuning/0.parquet
✅  data/held_out/0.parquet
✅  metadata/codes.parquet
✅  metadata/dataset.json
✅  metadata/subject_splits.parquet


## 2. Schéma MEDS

In [15]:
# Charger le split train
train = pd.read_parquet(OUTPUT_DIR / "data/train/0.parquet")

print("Types des colonnes :")
print(train.dtypes)
print()

# Vérifications de conformité
checks = {
    "subject_id est int64": str(train["subject_id"].dtype) in ("int64", "Int64"),
    "time est datetime": "datetime" in str(train["time"].dtype),
    "code est string/object": str(train["code"].dtype) in ("object", "string", "str"),
    "numeric_value est float32": str(train["numeric_value"].dtype) == "float32",
    "4 colonnes exactement": len(train.columns) == 4,
}

for check, result in checks.items():
    print(f"{'✅' if result else '❌'}  {check}")

Types des colonnes :
subject_id                Int64
time             datetime64[us]
code                        str
numeric_value           float32
dtype: object

✅  subject_id est int64
✅  time est datetime
✅  code est string/object
✅  numeric_value est float32
✅  4 colonnes exactement


## 3. Aperçu des données d'un patient

In [16]:
# Prendre le premier patient et afficher son historique complet
first_patient = train["subject_id"].iloc[0]
patient_df = train[train["subject_id"] == first_patient]

print(f"Patient {first_patient} — {len(patient_df)} événements")
patient_df.head(20)

Patient 4056806253170987 — 263 événements


,subject_id,time,code,numeric_value
0,4056806253170987,NaT,GENDER//M,NaN
1,4056806253170987,NaT,DIAGNOSIS//KCD8//I251,NaN
2,4056806253170987,NaT,DIAGNOSIS//KCD8//I839,NaN
3,4056806253170987,NaT,DIAGNOSIS//KCD8//Others,NaN
4,4056806253170987,2090-01-01 00:00:01,MEDS_BIRTH,NaN
5,4056806253170987,2130-10-29 11:32:28,ED_REGISTRATION,NaN
6,4056806253170987,2130-10-29 11:53:28,ED_OUT,NaN
7,4056806253170987,2130-10-29 12:38:28,HOSPITAL_ADMISSION//Emergency department,NaN
8,4056806253170987,2130-10-29 12:38:28,INSURANCE//Others,NaN
9,4056806253170987,2130-10-29 12:38:28,MARITAL_STATUS//single,NaN


## 4. Événements statiques (time = NaT)

In [17]:
# Les événements statiques doivent être en tête du dossier de chaque patient
static = train[train["time"].isna()]
print(f"Nombre d'événements statiques : {len(static)}")
print()
print("Types d'événements statiques :")
print(static["code"].apply(lambda x: x.split("//")[0]).value_counts())

Nombre d'événements statiques : 3528

Types d'événements statiques :
code
DIAGNOSIS    2466
GENDER       1062
Name: count, dtype: int64


## 5. Codes — qualité

In [6]:
# Vérifier l'absence de //nan dans les codes
nan_codes = train[train["code"].str.contains("//nan", na=False)]
print(f"Codes contenant //nan : {len(nan_codes)} {'✅' if len(nan_codes) == 0 else '❌'}")

# Vérifier l'absence de codes UNKNOWN
unknown_codes = train[train["code"] == "UNKNOWN"]
print(f"Codes UNKNOWN : {len(unknown_codes)}")

print()
print("Préfixes de codes présents :")
train["code"].apply(lambda x: x.split("//")[0]).value_counts()

Codes contenant //nan : 0 ✅
Codes UNKNOWN : 0

Préfixes de codes présents :


code
CHARTEVENT            330589
LAB                   109595
OUTPUT                 28861
INPUT_START            21242
MEDICATION              4491
DIAGNOSIS               2466
PROCEDURE_ICD           1175
GENDER                  1062
HOSPITAL_ADMISSION      1062
MARITAL_STATUS          1062
ICU_ADMISSION           1062
ICU_DISCHARGE           1062
HOSPITAL_DISCHARGE      1062
INSURANCE               1061
MEDS_BIRTH              1001
ED_REGISTRATION          461
ED_OUT                   461
MEDS_DEATH               144
Name: count, dtype: int64

In [7]:
# Codes les plus fréquents
print("Top 15 codes les plus fréquents :")
train["code"].value_counts().head(15)

Top 15 codes les plus fréquents :


code
CHARTEVENT//001C_1021_25105//회/min    26787
CHARTEVENT//001C_1023_27310//회/min    26786
CHARTEVENT//001C_1013_25640//mmHg     25111
CHARTEVENT//001C_1014_27095//mmHg     25085
CHARTEVENT//001C_1012_24795//mmHg     25050
CHARTEVENT//001C_1016_25640//mmHg     21842
CHARTEVENT//001C_1015_24795//mmHg     21799
CHARTEVENT//001C_1017_27095//mmHg     21792
OUTPUT//001O_1481_25470//cc           21721
CHARTEVENT//001C_1026_26520//℃        21048
INPUT_START//001I_1315_26175//cc      17253
CHARTEVENT//001C_1003_24480//%        16045
CHARTEVENT//001C_1187_20770           11297
CHARTEVENT//001C_1187_20765           11100
CHARTEVENT//001C_1810_22385            8960
Name: count, dtype: int64

## 6. Distribution des splits

In [8]:
splits = pd.read_parquet(OUTPUT_DIR / "metadata/subject_splits.parquet")

counts = splits["split"].value_counts()
total = len(splits)

print(f"Total patients : {total}")
print()
for split_name, count in counts.items():
    pct = count / total * 100
    print(f"  {split_name:10} : {count:5} patients ({pct:.1f}%)")

# Vérification des proportions
print()
checks = {
    "train ~80%": 75 <= counts.get("train", 0) / total * 100 <= 85,
    "tuning ~10%": 8 <= counts.get("tuning", 0) / total * 100 <= 12,
    "held_out ~10%": 8 <= counts.get("held_out", 0) / total * 100 <= 12,
}
for check, result in checks.items():
    print(f"{'✅' if result else '❌'}  {check}")

Total patients : 1328

  train      :  1062 patients (80.0%)
  held_out   :   134 patients (10.1%)
  tuning     :   132 patients (9.9%)

✅  train ~80%
✅  tuning ~10%
✅  held_out ~10%


## 7. Métadonnées — codes.parquet

In [9]:
codes = pd.read_parquet(OUTPUT_DIR / "metadata/codes.parquet")

print(f"Codes uniques : {len(codes)}")
print(f"Codes avec description : {codes['description'].notna().sum()}")
print(f"Codes sans description : {codes['description'].isna().sum()}")
print()
print("Exemples de codes avec description :")
codes[codes["description"].notna()][["code", "description"]].head(10)

Codes uniques : 182
Codes avec description : 91
Codes sans description : 91

Exemples de codes avec description :


,code,description
0,CHARTEVENT//001C_1003_24480//%,SpO2 > 산소포화도
1,CHARTEVENT//001C_1012_24795//mmHg,SBP > 수축기혈압
2,CHARTEVENT//001C_1013_25640//mmHg,DBP > 이완기혈압
3,CHARTEVENT//001C_1014_27095//mmHg,mean BP(연동) > 평균혈압(연동)
4,CHARTEVENT//001C_1015_24795//mmHg,ASBP > 수축기혈압
5,CHARTEVENT//001C_1016_25640//mmHg,ADBP > 이완기혈압
6,CHARTEVENT//001C_1017_27095//mmHg,mean ABP > 평균혈압(연동)
7,CHARTEVENT//001C_1021_25105//회/min,HR > 심박동수
8,CHARTEVENT//001C_1023_27310//회/min,RR > 호흡수
9,CHARTEVENT//001C_1026_26520//℃,BT > 체온


## 8. Métadonnées — dataset.json

In [10]:
meta = json.loads((OUTPUT_DIR / "metadata/dataset.json").read_text())
for key, value in meta.items():
    print(f"  {key:20} : {value}")

  dataset_name         : K-MIMIC-MEDS
  dataset_version      : 0.1.0
  etl_name             : kmimic-meds
  etl_version          : 0.1.0
  meds_version         : 0.3.3
  created_at           : 2026-04-14T06:42:01.906210+00:00


## 9. Statistiques globales

In [11]:
# Charger tous les splits
all_dfs = []
for split_name in ["train", "tuning", "held_out"]:
    df = pd.read_parquet(OUTPUT_DIR / f"data/{split_name}/0.parquet")
    df["split"] = split_name
    all_dfs.append(df)

full = pd.concat(all_dfs, ignore_index=True)

print("=== Statistiques globales ===")
print(f"Total événements     : {len(full):,}")
print(f"Total patients       : {full['subject_id'].nunique():,}")
print(f"Événements statiques : {full['time'].isna().sum():,}")
print(f"Événements dynamiques: {full['time'].notna().sum():,}")
print(f"Avec valeur numérique: {full['numeric_value'].notna().sum():,}")
print()
print("Événements par split :")
print(full.groupby("split").agg(
    evenements=("code", "count"),
    patients=("subject_id", "nunique")
))

=== Statistiques globales ===
Total événements     : 639,454
Total patients       : 1,328
Événements statiques : 4,419
Événements dynamiques: 635,035
Avec valeur numérique: 605,941

Événements par split :
          evenements  patients
split                         
held_out       69278       134
train         507919      1062
tuning         62257       132


## 10. Résumé de validation

In [12]:
print("=== RÉSUMÉ DE VALIDATION ===")
print()

all_checks = {
    "Fichiers présents"          : all((OUTPUT_DIR / f).exists() for f in expected),
    "subject_id est int64"       : str(train["subject_id"].dtype) in ("int64", "Int64"),
    "time est datetime"          : "datetime" in str(train["time"].dtype),
    "numeric_value est float32"  : str(train["numeric_value"].dtype) == "float32",
    "Pas de //nan dans les codes": len(train[train["code"].str.contains("//nan", na=False)]) == 0,
    "Événements statiques en NaT": train[train["time"].isna()]["code"].str.startswith(("GENDER", "DIAGNOSIS")).all(),
    "Splits 80/10/10"            : all(checks.values()),
    "codes.parquet enrichi"      : codes["description"].notna().sum() > 0,
    "dataset.json présent"       : (OUTPUT_DIR / "metadata/dataset.json").exists(),
}

for check, result in all_checks.items():
    print(f"{'✅' if result else '❌'}  {check}")

print()
passed = sum(all_checks.values())
total_checks = len(all_checks)
print(f"Résultat : {passed}/{total_checks} vérifications passées")

=== RÉSUMÉ DE VALIDATION ===

✅  Fichiers présents
✅  subject_id est int64
✅  time est datetime
✅  numeric_value est float32
✅  Pas de //nan dans les codes
✅  Événements statiques en NaT
✅  Splits 80/10/10
✅  codes.parquet enrichi
✅  dataset.json présent

Résultat : 9/9 vérifications passées
